# fst2framegraph Colab runner

Run each cell in order. Upload the project zip when prompted if the notebook is not already inside an extracted project directory, then upload the OxCCAL CSV when prompted.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload the fst2framegraph project zip.')
    project_dir = Path('/content/fst2framegraph')
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall(project_dir)
    candidates = [p for p in project_dir.rglob('pyproject.toml')]
    if not candidates:
        raise FileNotFoundError('The uploaded zip does not contain pyproject.toml.')
    root = candidates[0].parent
    os.chdir(root)

wheel = root / 'wheels' / 'sentencepiece-0.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
if not wheel.exists():
    subprocess.check_call([sys.executable, 'scripts/fetch_wheels.py'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', str(wheel)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--find-links=wheels/', '-e', '.'])

try:
    import frame_semantic_transformer  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, 'scripts/install_colab_fst.py'])

print(f'Project ready at {Path.cwd()}')

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_names:
    raise FileNotFoundError('Upload an OxCCAL-style CSV file.')
csv_path = Path(csv_names[0])
csv_path

In [ ]:
import pandas as pd
from run_pipeline import run_pipeline

result = run_pipeline(csv_path, output_root='colab_outputs')
display(pd.DataFrame([{
    'rows_in': result['rows_in'],
    'prepared_rows': result['rows_prepared'],
    'skipped_empty': result['rows_skipped_empty'],
    'documents': result['documents'],
    'graph_nodes': result['graph_nodes'],
    'graph_edges': result['graph_edges'],
    'lift_rows': result['lift_rows'],
}]))

lift = pd.read_csv(result['lift_path'])
display(lift)

print(result['summary_path'])
print(result['graph_path'])